In [ ]:
import os
import sys
sys.dont_write_bytecode = True

from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

# api keys
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
BASE_URL = os.getenv("BASE_URL")

# model selection
SMALL_MODEL_NAME_V1 = "mistralai/ministral-8b-2512"
SMALL_MODEL_NAME_V2 = "google/gemma-3-4b-it"
SMALL_MODEL_NAME_V3 = "meta-llama/llama-3.2-3b-instruct"
SMALL_MODEL_NAME_V4 = "meta-llama/llama-3.2-1b-instruct"

LARGE_MODEL_NAME_V1 = "qwen/qwen3.5-122b-a10b"
LARGE_MODEL_NAME_V2 = "qwen/qwen3.5-35b-a3b"

JUDGE_MODEL_NAME_V1 = "openai/gpt-5-mini"
JUDGE_MODEL_NAME_V2 = "anthropic/claude-sonnet-4.5"

SLM_AS_ROUTER_V1 = "mistralai/ministral-3b-2512"
SLM_AS_ROUTER_V2 = "meta-llama/llama-3.2-1b-instruct"

# concurrency level
RPS = 5

# additional configuration
DATA_LOCATION_PATH = "generated/slm_flow_df"
TOKENIZER_NAME = "o200k_base"
SPACY_NLP_MODEL = "en_core_web_lg"

# policies
SLM_AS_ROUTER_CONFIDENCE_THRESHOLD = 4

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

from core.data import RAGSyntheticDataset
from core.pipeline import RAGPipelineRunner, JScore, InferenceRecord
from core.messaging import LangchainMessageBuilder, PROMPT_REGISTRY
from core.router import (
    SLMRouterOutput,
    SLMRoutingPolicy,
    WeightedRuleBasedRoutingPolicy,
    WeightedRule
)
from core.tasks import RAGTaskPrediction
from core.utils import (
    get_evaluation_summary,
    compute_slm_routing_metrics,
    dump_to_csv
)

In [ ]:
dataset = RAGSyntheticDataset.from_files(DATA_LOCATION_PATH)

In [ ]:
dataset[:1]

In [ ]:
_base_rules = [
    WeightedRule(
        name="query_token_count",
        operator="ge",
        threshold=55,
        weight=0.10,
    ),
    WeightedRule(
        name="query_noun_chunk_count",
        operator="ge",
        threshold=7,
        weight=0.15,
    ),
    WeightedRule(
        name="query_avg_word_frequency",
        operator="le",
        threshold=3.0,
        weight=0.20,
    ),
    WeightedRule(
        name="avg_lexical_overlap",
        operator="le",
        threshold=0.10,
        weight=0.25,
    ),
]

reranking_task_rules = _base_rules + [
    WeightedRule(
        name="min_lexical_overlap",
        operator="le",
        threshold=0.05,
        weight=0.15,
    ),
    WeightedRule(
        name="documents_cosine_similarity",
        operator="ge",
        threshold=0.85,
        weight=0.25,
    ),
    WeightedRule(
        name="documents_count",
        operator="ge",
        threshold=10,
        weight=0.10,
    ),
]
compression_task_rules = _base_rules + [
    WeightedRule(
        name="total_context_token_count",
        operator="ge",
        threshold=5000,
        weight=0.30,
    ),
    WeightedRule(
        name="avg_chunk_token_count",
        operator="ge",
        threshold=600,
        weight=0.15,
    ),
    WeightedRule(
        name="relevant_documents_ratio",
        operator="le",
        threshold=0.20,
        weight=0.25,
    ),
]
message_builder = LangchainMessageBuilder.from_sequence(
    ("reranking", PROMPT_REGISTRY.reranking_inference, RAGTaskPrediction),
    ("context_compression", PROMPT_REGISTRY.context_compression_inference, RAGTaskPrediction),
    ("judge", PROMPT_REGISTRY.evaluation, JScore)
)
model_kwargs = {
    "api_key": OPENROUTER_API_KEY,
    "base_url": BASE_URL,
    "max_tokens": 4092,
    "temperature": 0.0,
    "rate_limiter": InMemoryRateLimiter(
        requests_per_second=RPS,
        check_every_n_seconds=0.1,
        max_bucket_size=RPS * 2
    )
}

### No routing: SLM

In [ ]:
small_model_only_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME_V1,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    routing_mode="slm",
    messages_builder=message_builder,
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
small_model_pipeline_result = await small_model_only_pipeline.arun(dataset)
dump_to_csv(small_model_pipeline_result, path="small_model_pipeline_result")

In [ ]:
small_model_pipeline_evaluation = await small_model_only_pipeline.aevaluate(small_model_pipeline_result)
dump_to_csv(small_model_pipeline_evaluation, path="small_model_pipeline_evaluation")

In [ ]:
small_model_only_pipeline_summary = get_evaluation_summary(small_model_pipeline_evaluation)
print(small_model_only_pipeline_summary)

### No routing: LLM

In [ ]:
large_model_only_pipeline = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME_V1,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    routing_mode="llm",
    messages_builder=message_builder,
    extractor_spacy_nlp=SPACY_NLP_MODEL,
    extractor_tokenizer_name=TOKENIZER_NAME,
    model_kwargs=model_kwargs
)

In [ ]:
large_model_pipeline_result = await large_model_only_pipeline.arun(dataset)
dump_to_csv(large_model_pipeline_result, path="large_model_pipeline_result")

In [ ]:
large_model_pipeline_evaluation = await large_model_only_pipeline.aevaluate(large_model_pipeline_result)
dump_to_csv(large_model_pipeline_evaluation, path="large_model_pipeline_evaluation")

In [ ]:
large_model_pipeline_summary = get_evaluation_summary(large_model_pipeline_evaluation)
print(large_model_pipeline_summary)

In [ ]:
_common_rule_based_kwargs = {
    "routing_mode": "dynamic",
    "messages_builder": message_builder,
    "dynamic_routing_policies": {
        "reranking": WeightedRuleBasedRoutingPolicy(
            *reranking_task_rules,
            min_triggers=3,
            cumulative_weights_threshold=0.65
        ),
        "context_compression": WeightedRuleBasedRoutingPolicy(
            *compression_task_rules,
            min_triggers=3,
            cumulative_weights_threshold=0.65
        )
    },
    "extractor_spacy_nlp": SPACY_NLP_MODEL,
    "extractor_tokenizer_name": TOKENIZER_NAME,
    "model_kwargs": model_kwargs
}

### Dynamic routing: Rule-based, ministral-8b-2512

In [ ]:
rule_based_pipeline__ministral_8b = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME_V1,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    **_common_rule_based_kwargs
)

In [ ]:
rule_based_pipeline__ministral_8b_result = await rule_based_pipeline__ministral_8b.arun(dataset)
dump_to_csv(rule_based_pipeline__ministral_8b_result, path="rule_based_pipeline__ministral_8b_result")

In [ ]:
rule_based_pipeline__ministral_8b_evaluation = await rule_based_pipeline__ministral_8b.aevaluate(rule_based_pipeline__ministral_8b_result)
dump_to_csv(rule_based_pipeline__ministral_8b_evaluation, path="rule_based_pipeline__ministral_8b_evaluation")

In [ ]:
rule_based__ministral_8b_summary = get_evaluation_summary(rule_based_pipeline__ministral_8b_evaluation)
print(rule_based__ministral_8b_summary)

In [ ]:
rule_based_pipeline__ministral_8b_routing_metrics = compute_slm_routing_metrics(rule_based_pipeline__ministral_8b_evaluation)
dump_to_csv([rule_based_pipeline__ministral_8b_routing_metrics], path="rule_based_pipeline_routing_metrics")
print(rule_based_pipeline__ministral_8b_routing_metrics)

### Dynamic routing: Rule-based, gemma-3-4b-it

In [ ]:
rule_based_pipeline__gemma3_4b = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME_V2,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    **_common_rule_based_kwargs
)

In [ ]:
rule_based_pipeline__gemma3_4b_result = await rule_based_pipeline__gemma3_4b.arun(dataset)
dump_to_csv(rule_based_pipeline__gemma3_4b_result, path="rule_based_pipeline__gemma3_4b_result")

In [ ]:
rule_based_pipeline__gemma3_4b_evaluation = await rule_based_pipeline__gemma3_4b.aevaluate(rule_based_pipeline__gemma3_4b_result)
dump_to_csv(rule_based_pipeline__gemma3_4b_evaluation, path="rule_based_pipeline__gemma3_4b_evaluation")

In [ ]:
rule_based__gemma3_4b_summary = get_evaluation_summary(rule_based_pipeline__gemma3_4b_evaluation)
print(rule_based__gemma3_4b_summary)

In [ ]:
rule_based_pipeline__gemma3_4b_routing_metrics = compute_slm_routing_metrics(rule_based_pipeline__gemma3_4b_evaluation)
dump_to_csv([rule_based_pipeline__gemma3_4b_routing_metrics], path="rule_based_pipeline__gemma3_4b_routing_metrics")
print(rule_based_pipeline__gemma3_4b_routing_metrics)

### Dynamic routing: Rule-based, Llama-3.2-3b-instruct

In [ ]:
rule_based_pipeline__llama3_3b = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME_V3,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    **_common_rule_based_kwargs
)

In [ ]:
rule_based_pipeline__llama3_3b_result = await rule_based_pipeline__llama3_3b.arun(dataset)
dump_to_csv(rule_based_pipeline__llama3_3b_result, path="rule_based_pipeline__llama3_3b_result")

In [ ]:
rule_based_pipeline__llama3_3b_evaluation = await rule_based_pipeline__llama3_3b.aevaluate(rule_based_pipeline__llama3_3b_result)
dump_to_csv(rule_based_pipeline__llama3_3b_evaluation, path="rule_based_pipeline__llama3_3b_evaluation")

In [ ]:
rule_based__llama3_3b_summary = get_evaluation_summary(rule_based_pipeline__llama3_3b_evaluation)
print(rule_based__llama3_3b_summary)

In [ ]:
rule_based_pipeline__llama3_3b_routing_metrics = compute_slm_routing_metrics(rule_based_pipeline__llama3_3b_evaluation)
dump_to_csv([rule_based_pipeline__llama3_3b_routing_metrics], path="rule_based_pipeline__llama3_3b_routing_metrics")
print(rule_based_pipeline__llama3_3b_routing_metrics)

### Dynamic routing: Rule-based, Llama-3.2-1b-instruct

In [ ]:
rule_based_pipeline__llama3_1b = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME_V4,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    **_common_rule_based_kwargs
)

In [ ]:
rule_based_pipeline__llama3_1b_result = await rule_based_pipeline__llama3_1b.arun(dataset)
dump_to_csv(rule_based_pipeline__llama3_1b_result, path="rule_based_pipeline__llama3_1b_result")

In [ ]:
rule_based_pipeline__llama3_1b_evaluation = await rule_based_pipeline__llama3_1b.aevaluate(rule_based_pipeline__llama3_1b_result)
dump_to_csv(rule_based_pipeline__llama3_1b_evaluation, path="rule_based_pipeline__llama3_1b_evaluation")

In [ ]:
rule_based__llama3_1b_summary = get_evaluation_summary(rule_based_pipeline__llama3_1b_evaluation)
print(rule_based__llama3_1b_summary)

In [ ]:
rule_based_pipeline__llama3_1b_routing_metrics = compute_slm_routing_metrics(rule_based_pipeline__llama3_1b_evaluation)
dump_to_csv([rule_based_pipeline__llama3_1b_routing_metrics], path="rule_based_pipeline__llama3_1b_routing_metrics")
print(rule_based_pipeline__llama3_1b_routing_metrics)

### Dynamic routing: SLM as a router

In [ ]:
slm_router_client = ChatOpenAI(model=SLM_AS_ROUTER_V1, **model_kwargs)

slm_routing_policy_messages_builder = LangchainMessageBuilder.from_sequence(
    ("slm_routing_policy", PROMPT_REGISTRY.slm_as_router, SLMRouterOutput)
)

slm_as_router_policy = SLMRoutingPolicy(
    client=slm_router_client,
    message_builder=slm_routing_policy_messages_builder,
    confidence_threshold=SLM_AS_ROUTER_CONFIDENCE_THRESHOLD
)

In [ ]:
_common_slm_based_kwargs = {
    "routing_mode": "dynamic",
    "messages_builder": message_builder,
    "dynamic_routing_policies": {
        "reranking": slm_as_router_policy,
        "context_compression": slm_as_router_policy
    },
    "extractor_spacy_nlp": SPACY_NLP_MODEL,
    "extractor_tokenizer_name": TOKENIZER_NAME,
    "model_kwargs": model_kwargs
}

In [ ]:
slm_router_based_pipeline__ministral_8b = RAGPipelineRunner(
    small_model=SMALL_MODEL_NAME_V1,
    large_model=LARGE_MODEL_NAME_V1,
    judge_model=JUDGE_MODEL_NAME_V1,
    **_common_slm_based_kwargs
)

In [ ]:
slm_router_based_pipeline__ministral_8b_result = await slm_router_based_pipeline__ministral_8b.arun(dataset)
dump_to_csv(slm_router_based_pipeline__ministral_8b_result, path="slm_router_based_pipeline__ministral_8b_result")

In [ ]:
slm_router_based_pipeline__ministral_8b_evaluation = (
    await slm_router_based_pipeline__ministral_8b.aevaluate(slm_router_based_pipeline__ministral_8b_result)
)
dump_to_csv(slm_router_based_pipeline__ministral_8b_evaluation, path="slm_router_based_pipeline__ministral_8b_evaluation")

In [ ]:
slm_router_based_pipeline__ministral_8b_summary = get_evaluation_summary(slm_router_based_pipeline__ministral_8b_evaluation)
print(slm_router_based_pipeline__ministral_8b_summary)

In [ ]:
slm_router_based_pipeline__ministral_8b_routing_metrics = compute_slm_routing_metrics(slm_router_based_pipeline__ministral_8b_evaluation)
dump_to_csv([slm_router_based_pipeline__ministral_8b_routing_metrics], path="slm_router_based_pipeline__ministral_8b_routing_metrics")
print(slm_router_based_pipeline__ministral_8b_routing_metrics)